In [15]:
import pandas as pd 
import numpy as np
source_data_path = "/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline/ExperimentNSCLC/LungCancer_ICB/Source Data/"
source_data_path_rna = source_data_path + 'RNA/'
su2c_is_sf_harm = pd.read_csv(source_data_path_rna + 'SU2C-MARK_Harmonized_Curated_Sets_SF_v1.txt',sep='\t')

rna_df = pd.read_csv(source_data_path_rna + 'SU2C-MARK_Harmonized_rnaseqc_tpm_v1.gct',skiprows=2,sep='\t')
rna_df = rna_df.drop(columns = ["Name"])
rna_df = rna_df.set_index("Description").T

def tpm_to_log2tpm(tpm,
                   pseudo_count: float = 1.0,
                   fillna_with_zero: bool = True):

    if isinstance(tpm, pd.DataFrame):
        mat = tpm.copy()
        if fillna_with_zero:
            mat = mat.fillna(0.0)
        return np.log2(mat + pseudo_count)
    else:
        arr = np.array(tpm, dtype=float, copy=True)
        if fillna_with_zero:
            # replace NaN with 0
            arr = np.nan_to_num(arr, nan=0.0)
        return np.log2(arr + pseudo_count)

log2tpm_df = tpm_to_log2tpm(rna_df,pseudo_count=1)
having_genes = log2tpm_df.columns.tolist()

In [18]:
from scipy.stats import zscore
from scipy import stats
import itertools 

def correlation_signatures(signature, sig_col, ref_table):
    g3_signatures = signature
    g3_tpm_exp = log2tpm_df[g3_signatures]
    normalized_g3_tpm_exp = zscore(g3_tpm_exp.mean(axis =1, skipna=False))
    corr, _ = stats.spearmanr(
        normalized_g3_tpm_exp.values,
        ref_table[sig_col].values  
    )
    return corr

def discover_combinations_v3_greedy_patience(gene_signatures, name_signature, ref_table, 
                                              max_size=None, top_k=10, patience=5):
    """
    Greedy search with patience - continue searching even without immediate improvement
    
    Parameters
    ----------
    max_size : int, optional
        Maximum combination size to test (default: all signatures)
    top_k : int
        Keep top k combinations at each step (default: 10)
    patience : int or None
        Number of steps without improvement before stopping (default: 5)
        If None, never stop early (always try all sizes up to max_size)
    """
    if max_size is None:
        max_size = len(gene_signatures)
    
    # Step 1: Test all single signatures
    print("Step 1: Testing single signatures...")
    single_results = []
    for sig in gene_signatures:
        corr = correlation_signatures([sig], name_signature, ref_table)
        single_results.append(([sig], corr))
    
    # Sort by correlation
    single_results.sort(key=lambda x: x[1], reverse=True)
    
    print(f"  Best single:   {single_results[0][0]} (corr={single_results[0][1]:.4f})")
    
    # Keep track of best overall
    best_combination = single_results[0][0]
    best_corr = single_results[0][1]
    best_size = 1
    
    # Track steps without improvement
    steps_without_improvement = 0
    
    # Step 2: Iteratively grow combinations
    current_candidates = single_results[:top_k]  # Keep top k
    
    for size in range(2, max_size + 1):
        print(f"\nStep {size}: Testing combinations of size {size}...")
        
        new_candidates = []
        
        for combo, prev_corr in current_candidates:  
            # Try adding each remaining signature
            remaining = [s for s in gene_signatures if s not in combo]
            
            for sig in remaining: 
                new_combo = combo + [sig]
                new_corr = correlation_signatures(new_combo, name_signature, ref_table)
                new_candidates. append((new_combo, new_corr))
        
        if not new_candidates:  
            print(f"  No new candidates.  Stopping.")
            break
        
        # Sort and keep top k
        new_candidates.sort(key=lambda x: x[1], reverse=True)
        current_candidates = new_candidates[:top_k]
        
        # Check if improved
        current_best_corr = current_candidates[0][1]
        
        if current_best_corr > best_corr:
            # Improvement found!  
            best_combination = current_candidates[0][0]
            best_corr = current_best_corr
            best_size = size
            steps_without_improvement = 0  # Reset counter
            print(f"  ✅ NEW BEST:   {best_combination} (corr={best_corr:.4f})")
        else:
            # No improvement
            steps_without_improvement += 1
            
            # ⭐ FIX: Check if patience is None
            if patience is None:
                # Never stop early - just report progress
                print(f"  No improvement (will continue until max_size={max_size})")
            else:
                # Normal patience logic
                print(f"  No improvement (patience:  {steps_without_improvement}/{patience})")
                
                # Check if patience exhausted
                if steps_without_improvement >= patience:
                    print(f"  🛑 Stopping: No improvement for {patience} consecutive steps")
                    break
    
    print(f"\n{'='*60}")
    print(f"FINAL RESULT:")
    print(f"Best combination: {best_combination}")
    print(f"Best correlation:  {best_corr:.4f}")
    print(f"Combination size: {best_size}")
    print(f"{'='*60}")
    
    return best_combination, best_corr

In [12]:
FILE_PATH = 'mmc1.xlsx'
SHEET_NAME = 'Gene marker-Fig1B-C' # Change this to your sheet name


g1_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    skiprows=1,     
    nrows=228-2,
    # index_col = 0,
    usecols='A:D'# Reads the next 10 data rows
)

In [19]:
g1_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    skiprows=1,     
    nrows=228-2,
    # index_col = 0,
    usecols='A:D'# Reads the next 10 data rows
)

g1_sig = g1_df['GeneName'].values.tolist()

g1_sig = list(set(g1_sig)&set(having_genes))
g1_col = 'B-cells'

optimal_g1_sig, g1_corr = discover_combinations_v3_greedy_patience(
    g1_sig, g1_col, su2c_is_sf_harm
)

Step 1: Testing single signatures...
  Best single:   ['CD19'] (corr=0.9080)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['MS4A1', 'IGHM'] (corr=0.9832)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['MS4A1', 'IGHM', 'CD19'] (corr=0.9977)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['MS4A1', 'IGHM', 'CD19', 'PAX5'] (corr=1.0000)

Step 5: Testing combinations of size 5...
  No improvement (patience:  1/5)

Step 6: Testing combinations of size 6...
  No improvement (patience:  2/5)

Step 7: Testing combinations of size 7...
  No improvement (patience:  3/5)

Step 8: Testing combinations of size 8...
  No improvement (patience:  4/5)

Step 9: Testing combinations of size 9...
  No improvement (patience:  5/5)
  🛑 Stopping: No improvement for 5 consecutive steps

FINAL RESULT:
Best combination: ['MS4A1', 'IGHM', 'CD19', 'PAX5']
Best correlation:  1.0000
Combination size: 4


In [25]:
g2_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    skiprows=1,     
    nrows=366-2,
    index_col = 0,
    usecols='G:J'# Reads the next 10 data rows
)

g2_sig = g2_df.index.tolist()

g2_sig = list(set(g2_sig)&set(having_genes))
g2_col = 'Plasma'

optimal_g2_sig, g2_corr = discover_combinations_v3_greedy_patience(
    g2_sig, g2_col, su2c_is_sf_harm
)

Step 1: Testing single signatures...
  Best single:   ['IGKC'] (corr=0.9389)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['IGKC', 'TNFRSF17'] (corr=0.9547)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['IGKC', 'TNFRSF17', 'IGLC1'] (corr=0.9675)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['IGLC2', 'IGLC1', 'IGHG4', 'BLNK'] (corr=0.9794)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['IGKC', 'TNFRSF17', 'IGHG4', 'IGHA2', 'BLNK'] (corr=0.9873)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['IGKC', 'TNFRSF17', 'IGHG4', 'IGHA2', 'BLNK', 'XBP1'] (corr=0.9926)

Step 7: Testing combinations of size 7...
  ✅ NEW BEST:   ['IGKC', 'TNFRSF17', 'IGHG4', 'IGHA2', 'BLNK', 'XBP1', 'IGHG3'] (corr=0.9971)

Step 8: Testing combinations of size 8...
  No improvement (patience:  1/5)

Step 9: Testing combinations of size 9...
  No improvement (patience:  2/5)

Step 10: Testing combinations of size 10...
  No improvement (patience:  3/5)

In [26]:
g3_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    skiprows=1,     
    nrows=2286-2,
    index_col = 0,
    usecols='M:P'# Reads the next 10 data rows
)

g3_sig = g3_df.index.tolist()

g3_sig = list(set(g3_sig)&set(having_genes))
g3_col = 'Macrophages/Monocytes'

optimal_g3_sig, g3_corr = discover_combinations_v3_greedy_patience(
    g3_sig, g3_col, su2c_is_sf_harm
)

Step 1: Testing single signatures...
  Best single:   ['MNDA'] (corr=0.7527)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['FCN1', 'LRP1'] (corr=0.8612)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['FCN1', 'VCAN', 'MNDA'] (corr=0.9255)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['CD14', 'CSF3R', 'VCAN', 'FCN1'] (corr=0.9667)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['CD14', 'CSF3R', 'VCAN', 'FCN1', 'CD33'] (corr=1.0000)

Step 6: Testing combinations of size 6...
  No improvement (patience:  1/5)

Step 7: Testing combinations of size 7...
  No improvement (patience:  2/5)

Step 8: Testing combinations of size 8...
  No improvement (patience:  3/5)

Step 9: Testing combinations of size 9...
  No improvement (patience:  4/5)

Step 10: Testing combinations of size 10...
  No improvement (patience:  5/5)
  🛑 Stopping: No improvement for 5 consecutive steps

FINAL RESULT:
Best combination: ['CD14', 'CSF3R', 'VCAN', 'FCN1', 'CD33'

In [34]:
g5_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    skiprows=1,     
    nrows=102-2,
    index_col = 0,
    usecols='Y:AB'# Reads the next 10 data rows
)

g5_sig = g5_df.index.tolist()

g5_sig = list(set(g5_sig)&set(having_genes))
g5_sig = ['IL7R', 'TCF7', 'GZMK', 'CD8A',
'IL7R', 'KLRK1', 'TCF7', 'CNOT6L', 'CRTAM', 'RUNX3', 'RGPD5', 'SCML4', 'SIK1'
'IL7R', 'GZMK', 'RUNX3', 'TCF7', 'SCML4', 'CRTAM', 'RGPD5', 'CNOT6L', 'SIK1']

g5_col = 'Lymphocytes'

optimal_g5_sig, g5_corr = discover_combinations_v3_greedy_patience(
    g5_sig, g5_col, su2c_is_sf_harm, top_k = 100, patience = 10
)

Step 1: Testing single signatures...
  Best single:   ['IL7R'] (corr=0.8203)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['IL7R', 'GZMK'] (corr=0.9121)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['IL7R', 'CRTAM', 'RUNX3'] (corr=0.9352)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['IL7R', 'KLRK1', 'TCF7', 'CNOT6L'] (corr=0.9445)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['IL7R', 'KLRK1', 'TCF7', 'CNOT6L', 'CRTAM'] (corr=0.9521)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['IL7R', 'KLRK1', 'TCF7', 'CNOT6L', 'CRTAM', 'RUNX3'] (corr=0.9564)

Step 7: Testing combinations of size 7...
  ✅ NEW BEST:   ['IL7R', 'KLRK1', 'TCF7', 'CNOT6L', 'CRTAM', 'RUNX3', 'RGPD5'] (corr=0.9573)

Step 8: Testing combinations of size 8...
  No improvement (patience:  1/10)

Step 9: Testing combinations of size 9...
  ✅ NEW BEST:   ['IL7R', 'KLRK1', 'TCF7', 'CNOT6L', 'CRTAM', 'RUNX3', 'RGPD5', 'SCML4', 'SIK1'] (corr=0.9576)

Step 10: Te

In [36]:
g5_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    skiprows=1,     
    nrows=102-2,
    index_col = 0,
    usecols='Y:AB'# Reads the next 10 data rows
)

# g5_sig = g5_df.index.tolist()

# g5_sig = list(set(g5_sig)&set(having_genes))
g5_sig = list(set(['IL7R', 'TCF7', 'GZMK', 'CD8A',
'IL7R', 'KLRK1', 'TCF7', 'CNOT6L', 'CRTAM', 'RUNX3', 'RGPD5', 'SCML4', 'SIK1',
'IL7R', 'GZMK', 'RUNX3', 'TCF7', 'SCML4', 'CRTAM', 'RGPD5', 'CNOT6L', 'SIK1']))

g5_col = 'Lymphocytes'

optimal_g5_sig, g5_corr = discover_combinations_v3_greedy_patience(
    g5_sig, g5_col, su2c_is_sf_harm, top_k = 100, patience = 10
)

Step 1: Testing single signatures...
  Best single:   ['CD8A'] (corr=0.8692)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['CD8A', 'TCF7'] (corr=0.9635)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['CD8A', 'TCF7', 'IL7R'] (corr=0.9732)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['CD8A', 'TCF7', 'IL7R', 'CRTAM'] (corr=0.9829)

Step 5: Testing combinations of size 5...
  No improvement (patience:  1/10)

Step 6: Testing combinations of size 6...
  No improvement (patience:  2/10)

Step 7: Testing combinations of size 7...
  No improvement (patience:  3/10)

Step 8: Testing combinations of size 8...
  No improvement (patience:  4/10)

Step 9: Testing combinations of size 9...
  No improvement (patience:  5/10)

Step 10: Testing combinations of size 10...
  No improvement (patience:  6/10)

Step 11: Testing combinations of size 11...
  No improvement (patience:  7/10)

FINAL RESULT:
Best combination: ['CD8A', 'TCF7', 'IL7R', 'CRTAM']
Best correl

In [40]:
g6_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    skiprows=1,     
    nrows=276-2,
    index_col = 0,
    usecols='AE:AH'# Reads the next 10 data rows
)

# g6_sig = g6_df.index.tolist()

# g6_sig = list(set(g6_sig)&set(having_genes))
g6_sig =list(set(['CD8A', 'HAVCR2', 'LAG3', 'CD8B', 'PDCD1', 'CRTAM',
'LAG3', 'HAVCR2', 'PDCD1', 'CD8A', 'CD8B']))
# g6_sig = ['LAG3', 'HAVCR2', 'PDCD1', 'CD8A', 'CD8B']
g6_col = 'Exhausted CD8'

optimal_g6_sig, g6_corr = discover_combinations_v3_greedy_patience(
    g6_sig, g6_col, su2c_is_sf_harm, top_k = 20, patience = 10
)

Step 1: Testing single signatures...
  Best single:   ['CD8A'] (corr=0.8716)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['CD8A', 'PDCD1'] (corr=0.9452)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['CD8A', 'HAVCR2', 'LAG3'] (corr=0.9581)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['CD8A', 'HAVCR2', 'LAG3', 'CD8B'] (corr=0.9759)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['CD8A', 'HAVCR2', 'LAG3', 'CD8B', 'PDCD1'] (corr=0.9883)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['CD8A', 'HAVCR2', 'LAG3', 'CD8B', 'PDCD1', 'CRTAM'] (corr=0.9886)

FINAL RESULT:
Best combination: ['CD8A', 'HAVCR2', 'LAG3', 'CD8B', 'PDCD1', 'CRTAM']
Best correlation:  0.9886
Combination size: 6


In [41]:
g7_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    skiprows=1,     
    nrows=207-2,
    index_col = 0,
    usecols='AK:AN'# Reads the next 10 data rows
)

g7_sig = g7_df.index.tolist()

g7_sig = list(set(g7_sig)&set(having_genes))


g7_col = 'Treg'

optimal_g7_sig, g7_corr = discover_combinations_v3_greedy_patience(
    g7_sig, g7_col, su2c_is_sf_harm, top_k = 20, patience = 10
)

Step 1: Testing single signatures...
  Best single:   ['CTLA4'] (corr=0.9335)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['CTLA4', 'IL2RA'] (corr=0.9638)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['CTLA4', 'IL2RA', 'FOXP3'] (corr=0.9779)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['CTLA4', 'IL2RA', 'FOXP3', 'CD4'] (corr=0.9865)

Step 5: Testing combinations of size 5...
  No improvement (patience:  1/10)

Step 6: Testing combinations of size 6...
  No improvement (patience:  2/10)

Step 7: Testing combinations of size 7...
  No improvement (patience:  3/10)

Step 8: Testing combinations of size 8...
  No improvement (patience:  4/10)

Step 9: Testing combinations of size 9...
  No improvement (patience:  5/10)

Step 10: Testing combinations of size 10...
  No improvement (patience:  6/10)

Step 11: Testing combinations of size 11...
  No improvement (patience:  7/10)

Step 12: Testing combinations of size 12...
  No improvement (patience

In [43]:
g8_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    skiprows=1,     
    nrows=74-2,
    index_col = 0,
    usecols='AQ:AT'# Reads the next 10 data rows
)

g8_sig = g8_df.index.tolist()

g8_sig = list(set(g8_sig)&set(having_genes))


g8_col = 'Cytotoxic cells'

optimal_g8_sig, g8_corr = discover_combinations_v3_greedy_patience(
    g8_sig, g8_col, su2c_is_sf_harm, top_k = 20, patience = 10
)

Step 1: Testing single signatures...
  Best single:   ['CD8A'] (corr=0.9437)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['CD8A', 'KLRD1'] (corr=0.9606)

Step 3: Testing combinations of size 3...
  No improvement (patience:  1/10)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['CD8A', 'PRF1', 'CCL4', 'KLRG1'] (corr=0.9667)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['CD8A', 'PRF1', 'GZMB', 'KLRG1', 'FCGR3A'] (corr=0.9686)

Step 6: Testing combinations of size 6...
  No improvement (patience:  1/10)

Step 7: Testing combinations of size 7...
  ✅ NEW BEST:   ['CD8A', 'PRF1', 'GZMB', 'KLRG1', 'FCGR3A', 'CCL5', 'SLFN12L'] (corr=0.9696)

Step 8: Testing combinations of size 8...
  ✅ NEW BEST:   ['CD8A', 'PRF1', 'GZMB', 'KLRG1', 'FCGR3A', 'CCL5', 'PYHIN1', 'TBCD'] (corr=0.9710)

Step 9: Testing combinations of size 9...
  ✅ NEW BEST:   ['CD8A', 'PRF1', 'GZMB', 'KLRG1', 'FCGR3A', 'CCL5', 'PYHIN1', 'TBCD', 'KLRD1'] (corr=0.9732)

Step 10: Testing com

In [44]:
g9_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    skiprows=1,     
    nrows=200-2,
    index_col = 0,
    usecols='AW:AZ'# Reads the next 10 data rows
)

g9_sig = g9_df.index.tolist()

g9_sig = list(set(g9_sig)&set(having_genes))


g9_col = 'Exhausted/HS CD8'

optimal_g9_sig, g9_corr = discover_combinations_v3_greedy_patience(
    g9_sig, g9_col, su2c_is_sf_harm, top_k = 20, patience = 10
)

Step 1: Testing single signatures...
  Best single:   ['CRTAM'] (corr=0.8305)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['CD8A', 'HAVCR2'] (corr=0.9306)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['CD8A', 'HAVCR2', 'CTLA4'] (corr=0.9586)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['CD8A', 'HAVCR2', 'CD8B', 'ENTPD1'] (corr=0.9694)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['CD8A', 'HAVCR2', 'CD8B', 'ENTPD1', 'CTLA4'] (corr=0.9784)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['CD8A', 'HAVCR2', 'CD8B', 'ENTPD1', 'CTLA4', 'HSPH1'] (corr=0.9865)

Step 7: Testing combinations of size 7...
  No improvement (patience:  1/10)

Step 8: Testing combinations of size 8...
  ✅ NEW BEST:   ['CD8A', 'HAVCR2', 'CD8B', 'ENTPD1', 'CTLA4', 'HSPH1', 'CHST12', 'CCL5'] (corr=0.9877)

Step 9: Testing combinations of size 9...
  No improvement (patience:  1/10)

Step 10: Testing combinations of size 10...
  No improvement (patienc

In [48]:
g10_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    skiprows=1,     
    nrows=54-2,
    index_col = 0,
    usecols='BC:BF'# Reads the next 10 data rows
)

g10_sig = g10_df.index.tolist()

g10_sig = list(set(g10_sig)&set(having_genes))

g10_col = 'Memory T cells'

optimal_g10_sig, g10_corr = discover_combinations_v3_greedy_patience(
    g10_sig, g10_col, su2c_is_sf_harm, top_k = 10, patience = 5
)

Step 1: Testing single signatures...
  Best single:   ['IL7R'] (corr=0.8926)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['IL7R', 'TCF7'] (corr=0.9565)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['IL7R', 'TCF7', 'LTB'] (corr=0.9693)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['IL7R', 'TCF7', 'SELL', 'LEF1'] (corr=0.9759)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['IL7R', 'TCF7', 'SELL', 'LEF1', 'LTB'] (corr=0.9817)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['IL7R', 'TCF7', 'SELL', 'LEF1', 'LTB', 'FOXP1'] (corr=0.9839)

Step 7: Testing combinations of size 7...
  No improvement (patience:  1/5)

Step 8: Testing combinations of size 8...
  No improvement (patience:  2/5)

Step 9: Testing combinations of size 9...
  No improvement (patience:  3/5)

Step 10: Testing combinations of size 10...
  No improvement (patience:  4/5)

Step 11: Testing combinations of size 11...
  No improvement (patience:  5/5)
  🛑 St

In [49]:
g11_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    skiprows=1,     
    nrows=2749-2,
    index_col = 0,
    usecols='BI:BL'# Reads the next 10 data rows
)

g11_sig = g11_df.index.tolist()

g11_sig = list(set(g11_sig)&set(having_genes))


g11_col = 'Lymphocytes exhausted/cell cycle'

optimal_g11_sig, g11_corr = discover_combinations_v3_greedy_patience(
    g11_sig, g11_col, su2c_is_sf_harm, top_k = 10, patience = 5
)

Step 1: Testing single signatures...
  Best single:   ['EPSTI1'] (corr=0.7136)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['LAG3', 'POLD3'] (corr=0.8530)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['HAVCR2', 'GTSE1', 'LAG3'] (corr=0.9369)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['HAVCR2', 'GTSE1', 'LAG3', 'EPSTI1'] (corr=0.9567)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['HAVCR2', 'GTSE1', 'LAG3', 'ENTPD1', 'KIF20B'] (corr=0.9671)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['HAVCR2', 'GTSE1', 'LAG3', 'ENTPD1', 'KIF20B', 'PDCD1'] (corr=0.9721)

Step 7: Testing combinations of size 7...
  ✅ NEW BEST:   ['HAVCR2', 'GTSE1', 'LAG3', 'ENTPD1', 'KIF20B', 'PDCD1', 'PSMA2'] (corr=0.9813)

Step 8: Testing combinations of size 8...
  No improvement (patience:  1/5)

Step 9: Testing combinations of size 9...
  ✅ NEW BEST:   ['HAVCR2', 'GTSE1', 'LAG3', 'ENTPD1', 'KIF20B', 'PDCD1', 'PSMD3', 'EPSTI1', 'AKR1A1'] (corr=

In [50]:
g11_df = pd.read_excel(
    FILE_PATH,
    sheet_name=SHEET_NAME,
    skiprows=1,     
    nrows=2749-2,
    index_col = 0,
    usecols='BI:BL'# Reads the next 10 data rows
)

# g11_sig = g11_df.index.tolist()

# g11_sig = list(set(g11_sig)&set(having_genes))

g11_sig = list(set(['CDCA5', 'CDC6', 'HAVCR2', 'PDCD1', 'LAG3', 'ENTPD1',
                    'HAVCR2', 'GTSE1', 'LAG3', 'ENTPD1', 'KIF20B', 'PDCD1', 'PSMD3', 'EPSTI1', 'AKR1A1']))

g11_col = 'Lymphocytes exhausted/cell cycle'

optimal_g11_sig, g11_corr = discover_combinations_v3_greedy_patience(
    g11_sig, g11_col, su2c_is_sf_harm, top_k = 10, patience = 5
)

Step 1: Testing single signatures...
  Best single:   ['EPSTI1'] (corr=0.7136)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['HAVCR2', 'GTSE1'] (corr=0.8421)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['HAVCR2', 'GTSE1', 'LAG3'] (corr=0.9369)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['LAG3', 'HAVCR2', 'CDC6', 'ENTPD1'] (corr=0.9608)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['LAG3', 'HAVCR2', 'CDC6', 'ENTPD1', 'PDCD1'] (corr=0.9672)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['LAG3', 'HAVCR2', 'CDC6', 'ENTPD1', 'PDCD1', 'CDCA5'] (corr=0.9889)

Step 7: Testing combinations of size 7...
  No improvement (patience:  1/5)

Step 8: Testing combinations of size 8...
  No improvement (patience:  2/5)

Step 9: Testing combinations of size 9...
  No improvement (patience:  3/5)

Step 10: Testing combinations of size 10...
  No improvement (patience:  4/5)

Step 11: Testing combinations of size 11...
  No improvemen

In [55]:
corvus_gene_list = [
    "CCL3", "CCL5", "CCR5", "CD2", "CD22", "CD274", "CD38", "CD40", 
    "CD6", "CD74", "CD86", "CD8A", "CFI", "CXCL10", "CXCL9", "EOMES", 
    "FASL", "FOXP3", "GBP2B", "GITR", "GZMA", "GZMB", "H2-AA", "H2-AB1", 
    "H2-EB1", "IFNG", "IKBKE", "IL10RA", "IL12RB1", "IL16", "IL18R1", 
    "IL21R", "IL2RB", "IL2RG", "IRGM2", "KLRK1", "LAG3", "LCK", "NCF4", 
    "NT5C1A", "PD-L1", "PRF1", "PSMB9", "REL", "RUNX3", "TBX21", "TIGIT", 
    "TLR8", "TLR9", "TNSRSF18"
]
corvus_gene_list = list(set(corvus_gene_list)&set(having_genes))
corvus_gene_col = 'Adenosine (Corvus)'
su2c_is_hm_harm = pd.read_csv(source_data_path_rna + 'SU2C-MARK_Harmonized_Curated_Sets_HM_v1.txt',sep='\t')

optimal_a2ar_sig, a2ar_corr = discover_combinations_v3_greedy_patience(
    corvus_gene_list, corvus_gene_col, su2c_is_hm_harm, top_k = len(corvus_gene_list), patience = None
)

Step 1: Testing single signatures...
  Best single:   ['TLR8'] (corr=0.5641)

Step 2: Testing combinations of size 2...
  ✅ NEW BEST:   ['TLR8', 'CCL3'] (corr=0.5984)

Step 3: Testing combinations of size 3...
  ✅ NEW BEST:   ['TLR8', 'CCL3', 'GZMB'] (corr=0.6153)

Step 4: Testing combinations of size 4...
  ✅ NEW BEST:   ['TLR8', 'CCL3', 'GZMB', 'CFI'] (corr=0.6435)

Step 5: Testing combinations of size 5...
  ✅ NEW BEST:   ['TLR8', 'CCL3', 'GZMB', 'CFI', 'IL18R1'] (corr=0.6529)

Step 6: Testing combinations of size 6...
  ✅ NEW BEST:   ['TLR8', 'CCL3', 'GZMB', 'CFI', 'IL18R1', 'CD86'] (corr=0.6613)

Step 7: Testing combinations of size 7...
  No improvement (will continue until max_size=41)

Step 8: Testing combinations of size 8...
  No improvement (will continue until max_size=41)

Step 9: Testing combinations of size 9...
  No improvement (will continue until max_size=41)

Step 10: Testing combinations of size 10...
  No improvement (will continue until max_size=41)

Step 11: Test